# RAGAS Benchmark — Sherlock Holmes Corpus

This notebook runs the **RAGAS quality evaluation** on RAG methods using the Sherlock Holmes corpus.

**Dataset**: Adventures of Sherlock Holmes (full text) chunked into overlapping passages.
Test questions are generated via `scripts/generate_testset.py` and evaluated on 4 RAGAS metrics:
- **Faithfulness** — Is the answer grounded in the retrieved context?
- **Answer Relevancy** — Does the answer address the question?
- **Context Precision** — Are the retrieved chunks relevant to the question?
- **Context Recall** — Do the retrieved chunks cover the ground truth?

In [ ]:
# --- Environment setup (must run first) ---
from nb_helpers.config import setup_environment, load_config
setup_environment()

from helpers.types import BenchmarkConfig

# --- Configuration ---
# Edit these variables before running

METHODS = ["vdb"]                  # Adapter methods to benchmark
MAX_QUESTIONS = 10                  # Number of test questions (0 = all)
TOP_K = 8                           # Chunks to retrieve per query
CHUNK_SIZE = 1200                   # Characters per chunk
CHUNK_OVERLAP = 200                 # Overlap between chunks
SKIP_INDEX = False                  # Set True to reuse existing indexes

config_dict = load_config(overrides={
    "methods": METHODS,
    "max_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})
config = BenchmarkConfig.from_dict(config_dict)

BENCHMARK = "ragas"
RUN_CONFIG = {
    "corpus": "sherlock_holmes",
    "num_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
}
print(f"Config: {len(METHODS)} methods, {MAX_QUESTIONS} questions, top_k={TOP_K}")

In [ ]:
# --- Download corpus (if not already present) ---
import subprocess, os
from nb_helpers.config import PROJECT_ROOT

corpus_file = PROJECT_ROOT / "corpus" / "adventures_of_sherlock_holmes.txt"
if not corpus_file.exists():
    print("Corpus not found — downloading from Project Gutenberg...")
    script = PROJECT_ROOT / "scripts" / "download_corpus.sh"
    subprocess.run(["bash", str(script)], check=True)
    print()
else:
    print(f"Corpus already present: {corpus_file}")

# Check test set
testset_file = PROJECT_ROOT / "testset" / "sherlock_holmes.json"
if not testset_file.exists():
    print("\nTest set not found. Generate it with:")
    print("  python scripts/generate_testset.py")
else:
    print(f"Test set present: {testset_file}")

In [ ]:
# --- Load Dataset ---
from nb_helpers.datasets import load_ragas_sherlock

chunks, testset = load_ragas_sherlock(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    max_questions=MAX_QUESTIONS,
)
print(f"\nSample question: {testset[0]['question']}")
print(f"Ground truth: {testset[0]['ground_truth'][:200]}...")

In [ ]:
# --- Run Benchmarks ---
from nb_helpers.pipeline import run_all_methods, save_results

all_results = await run_all_methods(
    methods=METHODS,
    chunks=chunks,
    testset=testset,
    config=config,
    skip_index=SKIP_INDEX,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Evaluate with RAGAS ---
from nb_helpers.metrics import evaluate_all_methods

all_results = await evaluate_all_methods(
    all_results,
    config=config,
    use_ragas=True,
    use_em_f1=False,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Summary Table ---
from nb_helpers.charts import summary_table

summary_table(all_results)

In [ ]:
# --- Charts ---
from nb_helpers.charts import ragas_bar_chart, latency_chart, cost_breakdown_chart

ragas_bar_chart(all_results, title="RAGAS Scores — Sherlock Holmes")
latency_chart(all_results, title="Avg Query Latency — Sherlock Holmes")
cost_breakdown_chart(all_results, title="Cost Breakdown — Sherlock Holmes")